In [ ]:
import torch
from torch import nn
import numpy as np

device = "mps"

In [ ]:
class PolyTorch(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(2, 1)

    def forward(self, x):
        return self.fc1(x)

In [ ]:
# criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()

model = PolyTorch().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.8, patience=5, min_lr=1e-3
)

func = lambda x: 2 * (x**2) + 5

X = np.linspace(-10, 10, 100) # might wanna shuffle this (np.random.permutation(N))
X_aug = np.stack([X**2, X], axis=-1)
# X_aug = np.stack([X, np.ones(100)], axis=-1)
y = func(X)

X = torch.as_tensor(X, dtype=torch.float).to(device)
X_aug = torch.as_tensor(X_aug, dtype=torch.float).to(device)
y = torch.as_tensor(y, dtype=torch.float).to(device)

epochs = 1000
model.train()
for _ in range(epochs):
    optimizer.zero_grad()

    pred_y = model(X_aug)

    loss = criterion(pred_y.squeeze(), y)
    print(loss)

    scheduler.step(loss.item())
    loss.backward()
    optimizer.step()


In [ ]:
list(model.parameters())